# **Generate video analysis after network training, evaluation, and refinement on the GUI**

In [13]:
import deeplabcut
from pathlib import Path
import time
import re
import os

# These conditions are mutually exclusive
kpms_trainVideos = False # If you want to sample key videos (e.g., last day for every mouse) to train a Keypoint Moseq model
refineVideos = False # If you have flagged videos where identity switches happened, try to rerun the video analysis with reID transformer. It performs unsupervised identity attributions

pcutoff = 0.8
tracking_method = "skeleton"

In [14]:
# Final model

path_config_file = r"C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions_topView_DLC\trainingAggression_trainingSessions_topView-MiguelRocha-2026-02-03\config.yaml"
shuffle = 2
iteration = 4

deeplabcut.export_model(path_config_file,
                        shuffle = shuffle,
                        iteration = iteration,
                        overwrite = False)


In [15]:
videos_path = Path(r"C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions")
avis = [f for f in videos_path.rglob("*topView_comp.avi")]
filtered_avis = [f for f in avis if "preTrain" not in str(f) and not any(f.parent.rglob(f"*topView_DLCtracking_pcutoff_{pcutoff}_{tracking_method}"))] # Filters through the preTrain videos and already processed videos

print(f"Number of videos to analyze: {len(filtered_avis)}")

Number of videos to analyze: 1


In [16]:
filtered_avis

[WindowsPath('C:/Users/stagk/OneDrive/Desktop/trainingAggression_MR/trainingSessions/trainingSessions/Maladaptive/Controls/20251110_mouse978528_trainingSessions/Day16/mouse978528_Day16_topView_comp.avi')]

In [17]:
# Filter the last day videos for every adaptive mice. The Maladaptive mice are filtered for the last day of the first week

if kpms_trainVideos:

    best = {}  # mouse_id -> (day_int, path)

    for p in avis:
        s = str(p)

        m_mouse = re.search(r"mouse(\d+)", s)
        m_day   = re.search(r"[\\/](Day)(\d+)", s)  # ...\Day02\...

        if not (m_mouse and m_day):
            continue

        mouse_id = m_mouse.group(1)
        day = int(m_day.group(2))

        # If path contains "Maladaptive", only keep single-digit Days
        if "Maladaptive" in s and day >= 10:
            continue

        if (mouse_id not in best) or (day > best[mouse_id][0]):
            best[mouse_id] = (day, p)

    last_day_paths = [t[1] for t in best.values()]
    last_day_by_mouse = {mid: t[1] for mid, t in best.items()}


In [18]:
if refineVideos:
    flaged_videos = [f for f in videos_path.rglob("*flag.txt")]
    refine_videos = [next(f.parent.glob("*topView*")) for f in flaged_videos]

In [19]:
if kpms_trainVideos:
    videos = last_day_paths
elif refineVideos:
    videos = refine_videos
else:
    videos = filtered_avis

for v in videos:

    start_T = time.time()

    print("\n") 
    print(f"============================= {v.name} =============================")

    destFolder = Path(f"{v.parent}/topView_DLCtracking_pcutoff_{pcutoff}_{tracking_method}")

    try:
        os.mkdir(destFolder)
    except FileExistsError:
        pass
    except OSError as e:
        print(f"Could not create folder {destFolder}: {e}")
        raise

    deeplabcut.analyze_videos(path_config_file,
                              v,
                              videotype=".avi",
                              shuffle=shuffle,
                              auto_track = True,
                              identity_only = True, # Implant keypoints are enough to distinguish mice. Bypasses identity estimation. This was possible by specifying keypoints only present in one mouse (implant -> Resident) during annotations
                              dynamic = (False, .5,10),
                              destfolder=destFolder)
    '''
    deeplabcut.transformer_reID(path_config_file,
                                str(v),
                                shuffle=shuffle,
                                videotype="avi",
                                track_method="skeleton",
                                n_triplets=6000,
                                train_epochs=200,
                                n_tracks = 2,
                                destfolder=destFolder)
    '''
                                               
    deeplabcut.filterpredictions(path_config_file, 
                                 str(v), 
                                 videotype=".avi", 
                                 shuffle=shuffle,
                                 filtertype="median",
                                 windowlength=11,
                                 track_method=tracking_method,
                                 destfolder=destFolder)

    deeplabcut.plot_trajectories(path_config_file, 
                                 str(v), 
                                 shuffle=shuffle, 
                                 filtered = True, 
                                 track_method=tracking_method,
                                 destfolder=destFolder)

    deeplabcut.create_labeled_video(path_config_file, 
                                    str(v), 
                                    videotype='.avi', 
                                    shuffle=shuffle, 
                                    filtered=True, 
                                    fastmode=True, 
                                    save_frames=False, 
                                    keypoints_only=False, 
                                    Frames2plot=None, 
                                    displayedbodyparts='all', 
                                    displayedindividuals='all',
                                    outputframerate=None, 
                                    draw_skeleton=True, 
                                    trailpoints=0, 
                                    displaycropped=False, 
                                    track_method=tracking_method,
                                    dotsize = 3,
                                    destfolder=destFolder,
                                    skeleton_color="red",
                                    confidence_to_alpha = True,
                                    plot_bboxes = False, 
                                    pcutoff = pcutoff,
                                    color_by = "individual")
    
    stop_T = time.time()

    print(f"Processing time: {((stop_T - start_T)/60):.2f} min\n")



============================= mouse978528_Day16_topView_comp.avi =============================
Analyzing videos with C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions_topView_DLC\trainingAggression_trainingSessions_topView-MiguelRocha-2026-02-03\dlc-models-pytorch\iteration-4\trainingAggression_trainingSessions_topViewFeb3-trainset99shuffle2\train\snapshot-best-100.pt
Using scorer: DLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle2_snapshot_best-100
Starting to analyze C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions\Maladaptive\Controls\20251110_mouse978528_trainingSessions\Day16\mouse978528_Day16_topView_comp.avi
Video metadata: 
  Overall # of frames:    62188
  Duration of video [s]:  1243.76
  fps:                    50.0
  resolution:             w=552, h=292

Running pose prediction with batch size 8


  0%|          | 16/62188 [00:00<06:43, 154.08it/s]d:\Anaconda3\envs\DEEPLABCUT\Lib\site-packages\deeplabcut\pose_estimation_pytorch\data\postprocessor.py:514: RuntimeWarning: invalid value encountered in cast
  heatmap_indices = np.rint(individual_keypoints).astype(int)
100%|██████████| 62188/62188 [19:42<00:00, 52.58it/s]


Processing...  C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions\Maladaptive\Controls\20251110_mouse978528_trainingSessions\Day16\mouse978528_Day16_topView_comp.avi
Loading From C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions\Maladaptive\Controls\20251110_mouse978528_trainingSessions\Day16\topView_DLCtracking_pcutoff_0.8_skeleton\mouse978528_Day16_topView_compDLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle2_snapshot_best-100.h5


100%|██████████| 62188/62188 [00:11<00:00, 5270.60it/s]


The tracklets were created (i.e., under the hood deeplabcut.convert_detections2tracklets was run). Now you can 'refine_tracklets' in the GUI, or run 'deeplabcut.stitch_tracklets'.
Processing...  C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions\Maladaptive\Controls\20251110_mouse978528_trainingSessions\Day16\mouse978528_Day16_topView_comp.avi


...

The videos are analyzed. Now your research can truly start!
You can create labeled videos with 'create_labeled_video'.
If the tracking is not satisfactory for some videos, consider expanding the training set. You can use the function 'extract_outlier_frames' to extract a few representative outlier frames.

Filtering with median model C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions\Maladaptive\Controls\20251110_mouse978528_trainingSessions\Day16\mouse978528_Day16_topView_comp.avi
Saving filtered csv poses!
Loading  C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions\Maladaptive\Controls\20251110_mouse978528_trainingSessions\Day16\mouse978528_Day16_topView_comp.avi and data.
Plots created! Please check the directory "plot-poses" within the video directory
Starting to process video: C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions\Maladaptive\Controls\20251110_mouse978528_tra

d:\Anaconda3\envs\DEEPLABCUT\Lib\site-packages\deeplabcut\utils\make_labeled_video.py:146: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  Dataframe.groupby(level="individuals", axis=1).size().values // 3


Duration of video [s]: 1243.78, recorded with 50.0 fps!
Overall # of frames: 62188 with cropped frame dimensions: 552 292
Generating frames and creating video.


100%|██████████| 62188/62188 [05:25<00:00, 191.03it/s]


Processing time: 26.61 min



In [ ]:
'''
import shutil

videos_path = Path(r"C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions")
avis = [f for f in videos_path.rglob("*refinedModel")]

for i in avis:
    shutil.rmtree(i)
'''

In [80]:
x= [f for f in videos_path.rglob("*flag1*")]
x

[WindowsPath('C:/Users/stagk/OneDrive/Desktop/trainingAggression_MR/trainingSessions/trainingSessions/20250825_mouse975833_trainingSessions/Day04_/flag1.txt'),
 WindowsPath('C:/Users/stagk/OneDrive/Desktop/trainingAggression_MR/trainingSessions/trainingSessions/20250929_mouse978772_trainingSessions/Day10/flag1.txt'),
 WindowsPath('C:/Users/stagk/OneDrive/Desktop/trainingAggression_MR/trainingSessions/trainingSessions/20251006_mouse1010819_trainingSessions/Day06/flag1.txt'),
 WindowsPath('C:/Users/stagk/OneDrive/Desktop/trainingAggression_MR/trainingSessions/trainingSessions/20251007_mouse1010823_trainingSessions/Day03/flag1.txt'),
 WindowsPath('C:/Users/stagk/OneDrive/Desktop/trainingAggression_MR/trainingSessions/trainingSessions/20251007_mouse1010823_trainingSessions/Day09/flag1.txt'),
 WindowsPath('C:/Users/stagk/OneDrive/Desktop/trainingAggression_MR/trainingSessions/trainingSessions/20251007_mouse1010823_trainingSessions/Day11/flag1.txt'),
 WindowsPath('C:/Users/stagk/OneDrive/Des